# 5.04 Web · Wikipedia · arXiv · PubMed Retriever

외부 지식 소스를 그대로 **retriever 인터페이스**로 감싸는 4종을 비교한다. 모두 `BaseRetriever` 규약을 따르므로 `.invoke("질의")` 한 줄이면 `List[Document]`가 나오고, 그대로 RAG 파이프라인에 꽂을 수 있다.

- `TavilySearchAPIRetriever` — Tavily 웹 검색 API (LLM-친화적 요약 포함)
- `WikipediaRetriever` — Wikimedia API 기반
- `ArxivRetriever` — arXiv API 기반, 논문 초록
- `PubMedRetriever` — NCBI E-utilities (PubMed)

## 학습 목표

- 4개 retriever의 공통 인터페이스와 **retrieve 단위 (웹 페이지 · 위키 문서 · 논문 메타)** 차이를 이해한다
- `top_k_results` / `k` / `doc_content_chars_max` 같은 하이퍼파라미터로 길이·개수 제어
- Wikipedia의 `lang="ko"`로 다국어 전환
- 4종을 묶어 한 질의에 대한 결과를 비교 (간이 멀티소스)

## 언제 쓰나

- **최신성**이 중요한 질의 (뉴스·업데이트): Tavily가 유일한 실시간 옵션
- 백과사전적 **개념 정의**: Wikipedia가 낮은 지연으로 안정적
- 학술·연구: arXiv(CS/물리/수학), PubMed(생의학)
- 에이전트에 **도구 여러 개**를 물려 질문 유형별로 라우팅


## 5.04.1 환경 설정

Tavily는 `TAVILY_API_KEY` 필요. 나머지는 키 없이 공개 API로 동작 (Wikipedia, arXiv, PubMed).
LLM/임베딩 호출이 없어 이 노트북은 **OPENAI_API_KEY 없이도** 통과한다.


In [ ]:
# !pip install -U langchain langchain-core langchain-community wikipedia arxiv xmltodict tavily-python

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 필요 (섹션 5.04.2, 5.04.6)"

# 나머지 라이브러리 import 검증
import wikipedia  # noqa: F401
import arxiv      # noqa: F401
import xmltodict  # noqa: F401  # PubMed가 XML 응답 파싱에 사용


## 5.04.2 기본 사용법 — Tavily 웹 검색

`TavilySearchAPIRetriever(k=...)`. 반환 `Document.metadata`에 `title`·`source`(URL)·`score` 등이 들어간다.


In [ ]:
from langchain_community.retrievers import TavilySearchAPIRetriever

tavily = TavilySearchAPIRetriever(k=3)
hits = tavily.invoke("LangChain v1 release notes")
for d in hits:
    print("-", d.metadata.get("title"))
    print("  URL   :", d.metadata.get("source"))
    print("  score :", d.metadata.get("score"))
    print("  body  :", d.page_content[:120], "...")


## 5.04.3 Wikipedia · Arxiv · PubMed

세 retriever 모두 **외부 라이브러리에 얇게 싸여** 있고, 공통 패턴으로 `top_k_results`, `doc_content_chars_max` 필드를 노출한다.

| Retriever | 주요 필드 | 반환 단위 |
|-----------|-----------|----------|
| `WikipediaRetriever` | `lang`, `top_k_results`, `load_all_available_meta` | 위키 페이지 (제목+요약 or 본문) |
| `ArxivRetriever` | `load_max_docs`, `get_full_documents`, `doc_content_chars_max` | 논문 초록(기본) / 본문(`get_full_documents=True`) |
| `PubMedRetriever` | `top_k_results`, `doc_content_chars_max` | 논문 초록 (MEDLINE) |


In [ ]:
from langchain_community.retrievers import WikipediaRetriever, ArxivRetriever, PubMedRetriever

wiki = WikipediaRetriever(lang="ko", top_k_results=2, doc_content_chars_max=800)
wiki_hits = wiki.invoke("트랜스포머 신경망")
print("--- Wikipedia (ko) ---")
for d in wiki_hits:
    print("-", d.metadata.get("title"), "|", d.page_content[:80], "...")

arxiv = ArxivRetriever(load_max_docs=2, doc_content_chars_max=500)
arxiv_hits = arxiv.invoke("retrieval augmented generation")
print("\n--- arXiv ---")
for d in arxiv_hits:
    print("-", d.metadata.get("Title") or d.metadata.get("title"))
    print("  ", d.page_content[:120], "...")

pubmed = PubMedRetriever(top_k_results=2, doc_content_chars_max=500)
pubmed_hits = pubmed.invoke("gut microbiome depression")
print("\n--- PubMed ---")
for d in pubmed_hits:
    print("-", d.metadata.get("Title") or d.metadata.get("title"))
    print("  ", d.page_content[:120], "...")


## 5.04.4 결과 비교 — 같은 질의, 4개 소스

각 retriever가 서로 다른 관점을 뽑아낸다. "`transformer architecture`" 같은 질의를 네 개에 모두 던져 결과 커버리지를 비교해 본다.


In [ ]:
query = "transformer architecture"

print(f"질의: {query!r}\n")

def header(name, count):
    print(f"[{name}] count={count}")

r_tavily = tavily.invoke(query)
header("Tavily", len(r_tavily))
for d in r_tavily[:2]:
    print("  -", d.metadata.get("title"))

r_wiki = WikipediaRetriever(lang="en", top_k_results=2, doc_content_chars_max=300).invoke(query)
header("Wikipedia(en)", len(r_wiki))
for d in r_wiki:
    print("  -", d.metadata.get("title"))

r_arxiv = ArxivRetriever(load_max_docs=2, doc_content_chars_max=200).invoke(query)
header("arXiv", len(r_arxiv))
for d in r_arxiv:
    print("  -", d.metadata.get("Title") or d.metadata.get("title"))

r_pubmed = PubMedRetriever(top_k_results=2, doc_content_chars_max=200).invoke(query)
header("PubMed", len(r_pubmed))
for d in r_pubmed:
    print("  -", d.metadata.get("Title") or d.metadata.get("title"))


## 5.04.5 `include_domains` · `exclude_domains` — Tavily 세부 필터

Tavily만의 유용한 옵션 몇 가지.
- `search_depth="basic" | "advanced"` — advanced는 더 긴 스냅샷 (비용 ↑)
- `include_domains=[...]` / `exclude_domains=[...]` — 도메인 허용/차단 리스트
- `include_generated_answer=True` — LLM 요약 응답 한 줄을 함께 반환 (일부 앱에 유용)

아래는 docs 사이트만 화이트리스트로 쓰는 예.


In [ ]:
tavily_docs_only = TavilySearchAPIRetriever(
    k=3,
    search_depth="advanced",
    include_domains=["python.langchain.com", "docs.langchain.com"],
)
for d in tavily_docs_only.invoke("create_agent middleware example"):
    print("-", d.metadata.get("source"))


## 5.04.6 (선택) `create_agent` 연동 — OpenAI 없이 통과

이 노트북은 OPENAI_API_KEY 없이도 실행되므로, 에이전트 연동은 **키가 있을 때만** 수행한다.
`create_retriever_tool`로 감싸 4개를 한 에이전트에 묶으면, 모델이 질문 유형을 보고 도구를 고른다.


In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain_classic.tools.retriever import create_retriever_tool
    from langchain.agents import create_agent

    tools = [
        create_retriever_tool(tavily, "web_search",  "최신 웹 정보를 찾는다. 뉴스·업데이트·실시간 이슈."),
        create_retriever_tool(wiki,   "wiki_search", "개념 정의·백과사전적 사실을 찾는다."),
        create_retriever_tool(arxiv,  "arxiv_search", "CS/수학/물리 논문 초록을 찾는다."),
        create_retriever_tool(pubmed, "pubmed_search", "의학·생명과학 논문을 찾는다."),
    ]
    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=tools,
        system_prompt="질문 유형에 맞는 검색 도구를 골라 먼저 호출한 뒤 결과를 근거로 답하라.",
    )
    r = agent.invoke({"messages": [{"role": "user", "content": "LangGraph checkpoint가 뭐야?"}]})
    print(r["messages"][-1].content[:500])
else:
    print("OPENAI_API_KEY 없음 → 에이전트 예제 스킵. retriever 자체는 위에서 이미 동작 확인됨.")


## 체크리스트

- [ ] Tavily 응답의 `metadata["source"]`는 실제 URL — 출처 인용에 그대로 활용 가능
- [ ] Wikipedia는 `lang`에 ISO 639-1 코드 (한국어는 `"ko"`, 일본어 `"ja"`). 일부 항목은 번역본 길이가 크게 다름
- [ ] `ArxivRetriever(get_full_documents=True)`는 PDF를 다운받아 파싱 — 느리고 트래픽 크다. 초록만으로 충분할 때가 많음
- [ ] PubMed는 `PUBMED_EMAIL`, `PUBMED_API_KEY` 환경변수로 **rate limit 상향 가능** (익명은 3 req/s)
- [ ] 웹·논문 retriever는 반환 `page_content`가 길 수 있어 **스플리터 후단**에 꽂는 것이 일반적

## 다음

- `05_vendor_managed.ipynb` — Amazon Kendra/KB, Azure AI Search, Vertex AI Search, Elasticsearch
- `05_advanced/05_agentic_rag.ipynb` — retriever 여러 개 묶은 에이전트 풀 예제

## 참고

- Tavily: https://tavily.com/
- LangChain retriever 통합 목록: https://python.langchain.com/docs/integrations/retrievers/
